# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zeeofficial01/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use Logistic Regression as the first machine-learning model.

The target is `is_declining`, which is a binary classification target from the Week-3 task framing. Logistic Regression is appropriate because it is a simple and interpretable classification model that provides a clear comparison against the Week-4 rule-based baseline.

The model will use the clean content-level signals identified in the previous work, including CTR, GSC impressions, average search position, and the CTR gap relative to the position-based reference.

I am choosing Logistic Regression before more complex models because the goal is to test whether a simple learned model can improve on the existing baseline, not to add complexity without evidence.

The model will be evaluated on the same data basis, split, and metric used for the baseline comparison.

## 2. Split design

I will use a grouped split by `client_hash_id` so that the same client does not appear in both the training and test sets.

This is a more honest test of whether the model can generalize to content from clients it did not train on. A random row-level split could allow the same client to appear in both sets and make the evaluation less independent.

I will keep the same data basis and evaluation approach used for the Week-4 baseline comparison.

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(dataset)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 9841378
    })
})


In [3]:
# Convert the Hugging Face dataset to a pandas DataFrame
df_raw = dataset["train"].to_pandas()

# Aggregate daily performance to content level, matching Week 4
df = (
    df_raw
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_sessions=("ga4_sessions", "sum"),
    )
)

# Calculate CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

print("Content-level rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())

Content-level rows: 331437

Columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ctr']


In [4]:
print("is_declining" in df.columns)
print(df.columns.tolist())

False
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ctr']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Target availability

The Week-2 task framing defined `is_declining_label` as the binary classification target, with F1-score as the success metric. However, the March 2026 content-performance dataset used for the Week-4 baseline does not contain `is_declining_label` or `trend_direction`.

After aggregating the March data to content level, there are 331,437 content items. Of these, 176,738 have complete values for the current modeling signals.

Because the original target labels are not present in this dataset, I will not create or infer `is_declining_label` from the available features. Doing so would change the original prediction task and could introduce a misleading evaluation.

For this Week-5 notebook, the available data will therefore be used for signal/model analysis rather than claiming a valid supervised decline-classification result.

In [5]:
# Check the data available for modeling
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# Check missing values in the modeling signals
print("\nMissing values:")
print(df[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ctr"
]].isna().sum())

Rows: 331437
Columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ctr']

Missing values:
gsc_impressions          0
gsc_clicks               0
gsc_avg_position    154699
ga4_sessions             0
ctr                 154699
dtype: int64


In [6]:
# Check how many rows have all required modeling features available

model_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ctr"
]

df_model = df.dropna(subset=model_features).copy()

print("Original rows:", len(df))
print("Rows available for modeling:", len(df_model))
print("Rows removed:", len(df) - len(df_model))

print("\nModeling columns:")
print(df_model[model_features].head())

Original rows: 331437
Rows available for modeling: 176738
Rows removed: 154699

Modeling columns:
    gsc_impressions  gsc_clicks  gsc_avg_position  ga4_sessions       ctr
3                 1           0          9.000000           0.0  0.000000
7               331           2         14.129210           0.0  0.006042
8                33           0          9.225529           0.0  0.000000
14              145           0          8.470926           0.0  0.000000
18              461           0         14.859827           0.0  0.000000


In [7]:
# Check whether the Week-2 target exists anywhere in the current notebook data

print("is_declining_label in df:", "is_declining_label" in df.columns)
print("is_declining_label in df_model:", "is_declining_label" in df_model.columns)

# Show all variables/columns that contain "declin" or "label"
print("\nRelevant columns in df:")
print([
    col for col in df.columns
    if "declin" in col.lower() or "label" in col.lower()
])

is_declining_label in df: False
is_declining_label in df_model: False

Relevant columns in df:
[]


In [8]:
# Recreate the Week-4 baseline scoring logic

# Create position buckets
df_model["position_bucket"] = pd.cut(
    df_model["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

# Calculate the reference CTR for each position bucket
position_reference = (
    df_model
    .groupby("position_bucket", observed=True)
    .agg(reference_ctr=("ctr", "mean"))
    .reset_index()
)

# Merge the reference CTR back into the data
df_model = df_model.merge(
    position_reference,
    on="position_bucket",
    how="left"
)

# Calculate CTR gap
df_model["ctr_gap"] = (
    df_model["reference_ctr"] - df_model["ctr"]
)

# CTR signal
df_model["ctr_signal"] = (
    df_model["ctr_gap"] > 0
).astype(int)

# Week-4 volume threshold
volume_threshold = df_model["gsc_impressions"].quantile(0.75)

# Volume signal
df_model["volume_signal"] = (
    df_model["gsc_impressions"] >= volume_threshold
).astype(int)

# Combined baseline score
df_model["baseline_score"] = (
    df_model["ctr_signal"] +
    df_model["volume_signal"]
)

print("Volume threshold:", volume_threshold)

print("\nBaseline score distribution:")
print(
    df_model["baseline_score"]
    .value_counts()
    .sort_index()
)

print("\nPosition reference CTR:")
print(position_reference)

Volume threshold: 1039.0

Baseline score distribution:
baseline_score
0     17167
1    124123
2     35448
Name: count, dtype: int64

Position reference CTR:
  position_bucket  reference_ctr
0             1-3       0.012399
1            4-10       0.004926
2           11-20       0.003211
3             21+       0.001928


In [9]:
# Summarize the baseline score groups

score_summary = (
    df_model
    .groupby("baseline_score")
    .agg(
        content_items=("content_hash_id", "count"),
        avg_impressions=("gsc_impressions", "mean"),
        median_impressions=("gsc_impressions", "median"),
        avg_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
        avg_position=("gsc_avg_position", "mean"),
        median_position=("gsc_avg_position", "median"),
        avg_ga4_sessions=("ga4_sessions", "mean"),
    )
    .reset_index()
)

print(score_summary)

   baseline_score  content_items  avg_impressions  median_impressions  \
0               0          17167       287.092037               196.0   
1               1         124123       554.265326                64.0   
2               2          35448      5837.621445              3013.0   

    avg_ctr  median_ctr  avg_position  median_position  avg_ga4_sessions  
0  0.037825    0.009174     13.977599         9.325669          3.909070  
1  0.000774    0.000000     17.783235         9.197017          4.932430  
2  0.001875    0.001449     10.731727         6.413518         16.588186  


In [10]:
# Compact baseline analysis table

baseline_comparison = (
    df_model
    .groupby("baseline_score")
    .agg(
        content_items=("content_hash_id", "count"),
        median_impressions=("gsc_impressions", "median"),
        median_ctr=("ctr", "median"),
        median_position=("gsc_avg_position", "median"),
        median_sessions=("ga4_sessions", "median"),
    )
    .reset_index()
)

print("Baseline signal comparison:")
display(baseline_comparison)

Baseline signal comparison:


,baseline_score,content_items,median_impressions,median_ctr,median_position,median_sessions
0,0,17167,196.0,0.009174,9.325669,1.0
1,1,124123,64.0,0.000000,9.197017,0.0
2,2,35448,3013.0,0.001449,6.413518,2.0


## 4. Errors and interpretation

A supervised model error analysis cannot be performed because the original `is_declining_label` target is not available in the March 2026 dataset.

Instead, I inspected how the Week-4 baseline score separates the available content-level signals.

The score-0 group contains 17,167 items, with median impressions of 196, median CTR of 0.009174, and median search position of 9.33.

The score-1 group contains 124,123 items, with median impressions of 64 and median CTR of 0.000000.

The score-2 group contains 35,448 items and has much higher search visibility, with median impressions of 3,013. Its median CTR is 0.001449 and median search position is 6.41.

The score-2 group therefore represents items where the baseline combines a CTR concern relative to the position reference with high search volume. This makes the score useful as a review-prioritization signal.

These results describe the behavior of the rule-based baseline. They do not show whether the items are actually declining because the original decline labels are unavailable.

## Self-check

- [x] Every section above is filled with markdown reasoning and supporting code
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are included
- [x] Claims use careful wording and do not imply that the missing decline labels were reconstructed
- [x] The Week-4 baseline logic was reproduced on the available complete March 2026 content-level data
- [x] Baseline signal groups were inspected and interpreted
- [ ] The original supervised Logistic Regression comparison could not be completed because `is_declining_label` is not present in the March 2026 dataset
- [ ] Final notebook committed under `work/notebooks/w05_model.ipynb`